# 💻 Lab 12 - Logical Agents: Core Inference Algorithms

> **Course**: Introduction to AI
> **Chapter**: 7 - Logical Agents

This lab focuses on implementing two core inference algorithms used by logical agents: **Truth Table Entailment (Model Checking)** and **Forward Chaining**. Through coding these algorithms, you'll learn how logical agents can deduce new facts from a known knowledge base using either exhaustive truth table evaluation or efficient rule-based propagation.

---

## 🎯 Objectives

- Implement `tt_entails` to determine whether a knowledge base entails a given query using model checking based on a dictionary/list representation.
- Implement `forward_chaining` to derive facts from a knowledge base using inference rules based on a dictionary/list representation.
- Learn how a simplified knowledge base (facts and rules) can be represented in Python dictionaries and lists.
- Validate your implementations using test cases that simulate logical inference.

---

## ⚙️ Logical Representation (Python Conventions)

We will use the following simplified symbolic conventions in Python:

- **Facts**: Represented by a Python dictionary where keys are fact names (strings) and values are the current state of the fact (can be boolean, numeric, string, etc.).
  - Example: `{"is_bird": True, "temperature": 20, "color": "red"}`

- **Rules**: Represented by a Python list of dictionaries, where each dictionary is a single rule.
  - Example: `[{"if": ..., "then": ...}, ...]`

- **Rule Structure**: Each rule dictionary must contain two keys:
  - `"if"`: A dictionary representing the premises. Keys are fact names (strings), and values are the **exact value** the corresponding fact must have for this premise to be satisfied. Function tests (like lambdas) are **not** used in this simplified version.
    - Example: `{"is_bird": True, "has_wings": True}`
  - `"then"`: A string representing the conclusion fact name. When all premises are met, this fact is inferred. In this system, inferred conclusions are typically boolean and set to `True` in the facts dictionary.
    - Example: `"can_fly"`

- **Knowledge Base (KB)**: The combination of the `facts` dictionary and the list of `rules`.

---

## 🛠️ Tasks

Complete the functions provided in the accompanying Python file (`lab12.py` or similar), based on the representation conventions described above.

### ✅ Task 1: Model Checking with Truth Tables

Complete the function `tt_entails(facts, rules, query)`:
   - **Purpose**: Determine if the knowledge base (given by `facts` and `rules`) logically entails the `query` fact using exhaustive truth table evaluation over relevant boolean facts and rule conclusions.
   - **Input**: `facts` (dict), `rules` (list of dicts), `query` (string).
   - **Returns**: `True` if for all possible assignments of boolean variables where the KB is true, the `query` fact is also true (or truthy in `facts`); otherwise `False`.

### ✅ Task 2: Inference by Forward Chaining

Complete the function `forward_chaining(facts, rules, query)`:
   - **Purpose**: Determine if the `query` fact can be inferred from the knowledge base (given by `facts` and `rules`) using the forward chaining inference procedure for rules with value-matching premises and boolean conclusions.
   - **Input**: `facts` (dict, modified in place during inference), `rules` (list of dicts), `query` (string).
   - **Returns**: `True` if the `query` fact is derived and added to the `facts` dictionary with a truthy value during the process; `False` otherwise.

---

## 🧪 Test Cases

Verify your implementations using the provided Knowledge Base defined in the accompanying Python file and the queries listed below.

### Test Queries

Run `tt_entails` and `forward_chaining` with the provided KB and the following queries:

1.  **Query**: `"can_fly"`
    * **Expected Result**: `True` (Should be inferred from `is_bird` and `has_wings`)

2.  **Query**: `"is_oviparous"`
    * **Expected Result**: `True` (Should be inferred from `lays_eggs`)

3.  **Query**: `"is_flying_bird"`
    * **Expected Result**: `True` (Should be inferred from `can_fly` and `is_oviparous`)

4.  **Query**: `"can_breathe_underwater"`
    * **Expected Result**: `False` (Requires `has_gills`, which is False in the provided facts)

5.  **Query**: `"is_fish"`
    * **Expected Result**: `False` (Requires `can_breathe_underwater`, which is False)

6.  **Query**: `"is_mammal"`
    * **Expected Result**: `False` (Not in facts and cannot be inferred by any rule)

---

## 🤔 Reflection Questions

- What is the main computational drawback of the `tt_entails` (truth table) approach, especially as the number of relevant boolean facts/conclusions grows?
- Which of the two implemented approaches (`tt_entails` or `forward_chaining`) is generally more scalable for large knowledge bases of this type, and why?


In [5]:
import itertools
from collections import deque

'''
Representation Conventions:

- Facts:
    A dictionary where keys are fact names (strings) and values can be of any type
    (e.g., boolean, integer, string). These represent the known state of the world.
    Example:
        facts = {
            "morning": True,
            "temperature": 20,  # Celsius
            "sunny": True
        }

- Rules:
    A list of dictionaries, where each dictionary represents a rule with an
    "if" part (conditions/premises) and a "then" part (conclusion).
    - The "if" part is a dictionary where keys are fact names (strings) and values are the
      **exact expected value** of the fact for the condition to hold. Function tests
      (like lambdas) are NOT used here.
    - The "then" part is a string representing the name of the fact to be inferred.
      When a rule fires, this inferred fact is asserted in the Knowledge Base
      with a value of True (specifically for boolean conclusions in the provided examples).
    Example:
        rules = [
            {"if": {"morning": True, "weekend": False}, "then": "workday"}
            # Rule based on temperature like > 25 would require a pre-calculated fact, e.g., "is_hot": True
        ]

- Entailment:
    Entailment (KB |= Q) means that in every model (assignment of truth values to
    symbols) where the Knowledge Base (KB, i.e., facts and rules) is true, the query (Q)
    must also be true. Two methods for checking entailment are provided:
    - Truth-Table Entailment (`tt_entails`): This method enumerates all possible
      truth assignments for the boolean variables and checks if the query is true
      in every assignment where the KB is true. This is sound and complete but can be
      computational expensive for a large number of boolean variables.
    - Forward Chaining (`forward_chaining`): This method is an inference procedure
      that derives new facts from existing ones using the rules until no new facts
      can be inferred. If the query is derived, then it is entailed. For the specific
      type of rules used here (Horn clauses leading to boolean conclusions, and premises
      only being direct value matches), forward chaining is sound and complete.

'''


def tt_entails(facts, rules, query):
    """
    Goal: Determine if a query is logically entailed by the given facts and rules using the Truth-Table Entailment method.
    This means checking if the query is true in every possible interpretation (model) where the facts are true and the rules hold.

    Input:
        facts: A dictionary representing the known facts (see Representation Conventions).
        rules: A list of dictionaries representing the rules (see Representation Conventions).

    Output:
        True if the query is entailed by the KB (facts + rules), False otherwise.

    Example Walkthrough:
        facts = {"A": True, "B": False}
        rules = [{"if": {"A": True, "B": False}, "then": "C"}]
        query = "C"

        ------------ Summary of Steps: ------------

        1. Collect all propositional variables into a set `bool_vars` from:
           - Fact keys
           - Rule premises (rule["if"].keys())
           - Rule conclusions (rule["then"])
           Example: bool_vars = {"A", "B", "C"}

        2. Sort the variables into a list `symbols` to ensure consistent iteration over models.
           Example: symbols = ["A", "B", "C"]

        3. Iterate over all possible truth assignments (models) for these variables.
           Use itertools.product([False, True], repeat=len(symbols)).
           Example: values = (False, False, False), model = {"A": False, "B": False, "C": False}

        4. For each model:
           - Check if the model satisfies all known facts (i.e., the model's value for each fact matches exactly).
             Example: model["A"] != facts["A"] → skip this model

           - If the model satisfies the facts, verify all rules:
             - For each rule, check if all premises are satisfied in the model.
             - If premises hold, ensure that the conclusion is also True in the model.
             Example: {"if": {"A": True, "B": False}, "then": "C"} → if model["A"] == True and model["B"] == False, then check model["C"] == True

           - If any rule fails in this model, it’s not a valid KB model → skip it.

        5. If the model satisfies both facts and rules, it is a valid model of the knowledge base.
           - Check if the query is True in this model.
             - If the query is False → return False (entailment fails)
             - If the query is True → continue checking other models

        6. After checking all models:
           - If every valid KB model has the query True → return True (entailment holds)


    Example Input:
        facts = {"is_bird": True, "has_wings": True}
        rules = [{"if": {"is_bird": True, "has_wings": True}, "then": "can_fly"}]
        query = "can_fly"

    Example Output:
        True
    """
    bool_vars = set(facts.keys())
    for rule in rules:
        bool_vars.update(rule["if"].keys())
        bool_vars.add(rule["then"])
    symbols = sorted(bool_vars)
    for values in itertools.product([False, True], repeat=len(symbols)):
        model = dict(zip(symbols, values))
        # Check if the model satisfies all known facts
        if any(fact not in model or model[fact] != facts[fact] for fact in facts):
            continue

        # Check if the model satisfies all rules
        for rule in rules:
            # Check if all premises are satisfied
            if all(model[premise] == rule["if"][premise] for premise in rule["if"]):
                # Check if the conclusion is True in the model
                if not model[rule["then"]]:
                    return False
        # Check if the query is True in this model
        if not model.get(query, False):
            return False
    return True


def forward_chaining(facts, rules, query):
    """
    Goal: Determine if a query can be inferred from the given facts and rules using the Forward Chaining inference procedure.
    This method iteratively applies rules to derive new facts—even those requiring False-valued premises—until the query is found
    (and True) or no new facts can be inferred.

    Input:
        facts: A dictionary representing the known facts (see Representation Conventions). This dictionary will be modified in place.
               Facts may be True or False.
        rules: A list of dictionaries representing the rules (see Representation Conventions). Each rule has:
               - "if": a dict of premise fact→expected_value (True or False)
               - "then": a single fact name to assert True when all premises match.
        query: A string representing the fact name to check if it can be inferred and is True.

    Output:
        True if the query is inferred (i.e., derived and set to True in facts), False otherwise.

    Example Walkthrough (with a False-valued premise):
        facts = {"P": True,  "Q": False}
        rules = [
            {"if": {"P": True,  "Q": False}, "then": "R"},
            {"if": {"R": True},                "then": "S"}
        ]
        query = "S"

        1. Initialize `agenda` with all known fact names:
           agenda = deque(["P", "Q"])

        2. Build:
           - triggers: map each premise-var to rules that test it:
               {"P": [Rule1], "Q": [Rule1], "R": [Rule2]}
           - count: map each rule ID to how many premises are unmet:
               {id(Rule1): 2, id(Rule2): 1}

        3. Process the agenda until empty:
           a. Dequeue p = "P".
              - Not the query.
              - For Rule1: expected_value for "P" is True; facts["P"] == True ⇒ decrement count[Rule1] to 1.
           b. Dequeue p = "Q".
              - Not the query.
              - For Rule1: expected_value for "Q" is False; facts["Q"] == False ⇒ decrement count[Rule1] to 0.
                All premises of Rule1 met:
                  • conclusion = "R"; not yet in facts ⇒ facts["R"]=True, enqueue "R".
           c. Dequeue p = "R".
              - Not the query.
              - For Rule2: expected_value for "R" is True; facts["R"] == True ⇒ decrement count[Rule2] to 0.
                All premises met:
                  • conclusion = "S"; not yet in facts ⇒ facts["S"]=True, enqueue "S".
           d. Dequeue p = "S".
              - p == query and facts["S"] is True ⇒ return True.

        4. If agenda empties without inferring query=True, return False.

    Example Input:
        facts = {"is_bird": True, "has_wings": True}
        rules = [{"if": {"is_bird": True, "has_wings": True}, "then": "can_fly"}]
        query = "can_fly"

    Example Output:
        True
    """
    agenda = deque(facts.keys())

    # Properly build triggers and count
    triggers = {}
    count = {}
    for rule in rules:
        rule_id = id(rule)
        count[rule_id] = len(rule["if"])
        for premise in rule["if"]:
            if premise not in triggers:
                triggers[premise] = []
            triggers[premise].append(rule)

    while agenda:
        p = agenda.popleft()
        if p == query and facts.get(p, False):
            return True
        for rule in triggers.get(p, []):
            expected_value = rule["if"][p]
            if facts.get(p, False) == expected_value:
                count[id(rule)] -= 1
                if count[id(rule)] == 0:
                    conclusion = rule["then"]
                    if conclusion not in facts:
                        facts[conclusion] = True
                        agenda.append(conclusion)
    return False


# --- Example with meaningful predicates (adapted for no function objects) ---
facts = {
    "is_bird": True,
    "has_wings": True,
    "lays_eggs": True,
    "has_gills": False,  # this one stays false
    # To include temperature > 25, we would need a fact like "is_hot": True/False
}

rules = [
    # if it is a bird AND has wings → it can fly
    {"if": {"is_bird": True, "has_wings": True}, "then": "can_fly"},
    # if it lays eggs → it is oviparous
    {"if": {"lays_eggs": True}, "then": "is_oviparous"},
    # if it can fly AND is oviparous → it is a flying bird
    {"if": {"can_fly": True, "is_oviparous": True}, "then": "is_flying_bird"},
    # if it has gills → it can breathe underwater
    {"if": {"has_gills": True}, "then": "can_breathe_underwater"},
    # if it can breathe underwater AND lays eggs → it is a fish
    {"if": {"can_breathe_underwater": True, "lays_eggs": True}, "then": "is_fish"},
    # Rule for hot day cannot be represented directly without function objects,
    # unless we add a pre-computed fact like "temperature_gt_25": True/False
    # For simplicity, removing the rule that used a lambda for temperature.
]

# Print the KB used for tests
print("Knowledge Base (KB) used for tests:")
print("Facts:", facts)
print("Rules:")
for i, rule in enumerate(rules):
    print(f"  Rule {i+1}: {rule}")
print("-" * 30)

# Goal: Demonstrate the usage of forward_chaining and tt_entails with the example KB.
# Input: The defined 'facts' dictionary, 'rules' list, and specific query strings.
# Output: Prints whether each query can be inferred using forward chaining and entailed using truth table entailment, followed by the boolean result.

# Queries we *can* infer:
print("--- Queries that are expected to be inferred/entailed ---")
for q in ["can_fly", "is_oviparous", "is_flying_bird"]:
    # Use a copy of facts for forward_chaining as it modifies the dictionary.
    print(f"with forward chaining__Can infer {q:20}? ->", forward_chaining(facts.copy(), rules, q))
    # Use a copy of facts for tt_entails. Note: tt_entails also modifies the facts copy by potentially adding inferred boolean facts.
    print(f"with Truth Table Entail__Can infer {q:20}? ->", tt_entails(facts.copy(), rules, q))

# Queries we *cannot* infer:
print("\n--- Queries that are expected NOT to be inferred/entailed ---")
for q in ["can_breathe_underwater", "is_fish", "is_mammal"]:
    # Use a copy of facts for forward_chaining.
    print(f"with forward chaining__Can infer {q:20}? ->", forward_chaining(facts.copy(), rules, q))
    # Use a copy of facts for tt_entails. Note: tt_entails also modifies the facts copy by potentially adding inferred boolean facts.
    print(f"with Truth Table Entail__Can infer {q:20}? ->", tt_entails(facts.copy(), rules, q))

Knowledge Base (KB) used for tests:
Facts: {'is_bird': True, 'has_wings': True, 'lays_eggs': True, 'has_gills': False}
Rules:
  Rule 1: {'if': {'is_bird': True, 'has_wings': True}, 'then': 'can_fly'}
  Rule 2: {'if': {'lays_eggs': True}, 'then': 'is_oviparous'}
  Rule 3: {'if': {'can_fly': True, 'is_oviparous': True}, 'then': 'is_flying_bird'}
  Rule 4: {'if': {'has_gills': True}, 'then': 'can_breathe_underwater'}
  Rule 5: {'if': {'can_breathe_underwater': True, 'lays_eggs': True}, 'then': 'is_fish'}
------------------------------
--- Queries that are expected to be inferred/entailed ---
with forward chaining__Can infer can_fly             ? -> True
with Truth Table Entail__Can infer can_fly             ? -> False
with forward chaining__Can infer is_oviparous        ? -> True
with Truth Table Entail__Can infer is_oviparous        ? -> False
with forward chaining__Can infer is_flying_bird      ? -> True
with Truth Table Entail__Can infer is_flying_bird      ? -> False

--- Queries that